# Attention Head Pruning

Impact of removing individual attention heads: ablation impact,
importance ranking, pruning tolerance, and output norm distribution.

## Why This Matters

Attention head pruning reveals how heads route information between positions. Understanding attention mechanics is central to reverse-engineering transformer circuits, as attention patterns determine what information each component can access and transform.

**Key references:**
- [Elhage et al. (2021) "A Mathematical Framework for Transformer Circuits"](https://transformer-circuits.pub/2021/framework/index.html) — Foundational framework for attention head and residual stream analysis
- [Olsson et al. (2022) "In-context Learning and Induction Heads"](https://arxiv.org/abs/2209.11895) — Discovers induction heads and training phase transitions

In [ ]:
import jax
import jax.numpy as jnp
from irtk import HookedTransformer, HookedTransformerConfig
from irtk.attention_head_pruning import (
    head_ablation_impact, head_importance_ranking,
    head_pruning_tolerance, head_output_norm_distribution,
    head_pruning_summary,
)

cfg = HookedTransformerConfig(
    n_layers=4, d_model=32, n_ctx=64, d_head=8, n_heads=4, d_vocab=100,
)
model = HookedTransformer(cfg)
key = jax.random.PRNGKey(42)
leaves, treedef = jax.tree.flatten(model)
new_leaves = []
for leaf in leaves:
    if isinstance(leaf, jnp.ndarray) and leaf.dtype == jnp.float32:
        key, subkey = jax.random.split(key)
        new_leaves.append(jax.random.normal(subkey, leaf.shape) * 0.1)
    else:
        new_leaves.append(leaf)
model = jax.tree.unflatten(treedef, new_leaves)
tokens = jnp.array([1, 15, 30, 45, 60, 75, 90])
print('Model ready')

## Head Ablation Impact

What happens when each head is removed?

In [ ]:
for layer in range(4):
    result = head_ablation_impact(model, tokens, layer=layer, position=-1)
    print(f"  Layer {layer} (most impactful: H{result['most_impactful_head']})")
    for h in result['per_head'][:3]:
        flag = ' ← CHANGES PRED' if h['prediction_changes'] else ''
        bar = '█' * int(h['mse_change'] * 10)
        print(f"    H{h['head']}: MSE={h['mse_change']:.4f} {bar}{flag}")

## Head Importance Ranking

Global ranking of all heads by ablation impact.

In [ ]:
result = head_importance_ranking(model, tokens, position=-1)
print(f"Most important: {result['most_important']}")
print(f"Least important: {result['least_important']}\n")
for h in result['ranking'][:8]:
    bar = '█' * int(h['importance'] * 10)
    print(f"  L{h['layer']}H{h['head']}: imp={h['importance']:.4f}, "
          f"norm={h['output_norm']:.4f} {bar}")

## Pruning Tolerance

How many heads can be removed before the prediction changes?

In [ ]:
result = head_pruning_tolerance(model, tokens, position=-1)
print(f"Heads prunable: {result['heads_prunable']} / {result['total_heads']}")
print(f"Tolerance: {result['pruning_tolerance']:.2%}")

## Output Norm Distribution

Identify dead or dominant heads by output norm.

In [ ]:
result = head_output_norm_distribution(model, tokens, position=-1)
print(f"Mean norm: {result['mean_norm']:.4f}")
print(f"Near-dead heads: {result['n_near_dead']}\n")
for h in sorted(result['per_head'], key=lambda x: x['output_norm'], reverse=True)[:8]:
    bar = '█' * int(h['output_norm'] * 10)
    print(f"  L{h['layer']}H{h['head']}: {h['output_norm']:.4f} {bar}")

## Pruning Summary

Combined head pruning analysis.

In [ ]:
result = head_pruning_summary(model, tokens, position=-1)
print(f"Most important: {result['most_important']}")
print(f"Least important: {result['least_important']}")
print(f"Prunable: {result['heads_prunable']}/{result['total_heads']} ({result['pruning_tolerance']:.0%})")
print(f"Near-dead: {result['n_near_dead']}")
print(f"Mean output norm: {result['mean_output_norm']:.4f}")